In [8]:
import json
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI()

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

In [2]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of currently popular movies.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {"type": "integer", "description": "The movie ID"}
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get the cast and crew credits for a movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {"type": "integer", "description": "The movie ID"}
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get a list of similar movies for a movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {"type": "integer", "description": "The movie ID"}
                },
                "required": ["movie_id"],
            },
        },
    },
]

In [3]:
def call_tool(name, args):
    if name == "get_popular_movies":
        url = f"{BASE_URL}/movies"
    elif name == "get_movie_details":
        url = f"{BASE_URL}/movies/{args['movie_id']}"
    elif name == "get_movie_credits":
        url = f"{BASE_URL}/movies/{args['movie_id']}/credits"
    elif name == "get_similar_movies":
        url = f"{BASE_URL}/movies/{args['movie_id']}/similar"
    else:
        return json.dumps({"error": f"Unknown tool: {name}"})
    return requests.get(url).text

In [4]:
def run_agent(user_message):
    messages = [
        {
            "role": "system",
            "content": "You are a movie expert. Use the provided tools to answer questions about movies. Always look up real data before answering.",
        },
        {"role": "user", "content": user_message},
    ]

    while True:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
        )
        msg = response.choices[0].message
        messages.append(msg)

        if msg.tool_calls:
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                print(f"[Tool Call] {tc.function.name}({args})")
                result = call_tool(tc.function.name, args)
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": result,
                    }
                )
        else:
            print(msg.content)
            return

In [5]:
run_agent("지금 인기 있는 영화가 무엇인지 알려줘")

[Tool Call] get_popular_movies({})
현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Mercy**
   - **개요**: 가까운 미래, 한 형사가 아내를 살해한 혐의로 재판을 받게 되고, 자신의 무죄를 입증하기 위해 고도로 발전한 AI 판사와 90분간의 대결을 벌입니다.
   - **평점**: 7.1
   - **개봉일**: 2026-01-20
   - ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

2. **28 Years Later: The Bone Temple**
   - **개요**: 주인공이 새로운 관계를 형성하면서 세상을 바꿀 수 있는 사건들이 펼쳐집니다.
   - **평점**: 7.199
   - **개봉일**: 2026-01-14
   - ![28 Years Later: The Bone Temple](https://image.tmdb.org/t/p/w780/kK1BGkG3KAvWB0WMV1DfOx9yTMZ.jpg)

3. **Les Orphelins**
   - **개요**: 어린 시절 친구가 갈등을 겪은 후, 그들이 사랑한 사람의 죽음을 조사하며 다시 팀을 이룬 이야기를 다룹니다.
   - **평점**: 6.043
   - **개봉일**: 2025-08-20
   - ![Les Orphelins](https://image.tmdb.org/t/p/w780/hP7mjZr2SVfjAorlRHTdV1XZmHY.jpg)

4. **A Woman Scorned**
   - **개요**: 여동생과 함께하는 시골 주말에 위협을 받게 되는 두 자매의 이야기를 그립니다.
   - **평점**: 6
   - **개봉일**: 2025-06-09
   - ![A Woman Scorned](https://image.tmdb.org/t/p/w780/dlOSBiNULMPzKIze84LDjvEN9z1.jpg)

5. **Deathstalker**
   -

In [6]:
run_agent("movie ID 550에 해당하는 영화가 무엇인지 알려줘")

[Tool Call] get_movie_details({'movie_id': 550})
영화 ID 550에 해당하는 영화는 **Fight Club**입니다.

- **개봉일**: 1999년 10월 15일
- **장르**: 드라마, 스릴러
- **줄거리**: 불면증에 시달리는 남자와 교묘한 비누 판매자가 원초적인 남성 공격성을 새로운 형태의 치료로 발산하게 됩니다. 이들의 아이디어는 전국적으로 퍼져 나가면서 언더그라운드 "파이트클럽"이 형성되지만, 이상한 인물이 개입하면서 모든 것이 통제를 벗어나게 됩니다.
- **상영 시간**: 139분
- **예산**: 63,000,000 달러
- **수익**: 100,853,753 달러
- **IMDb 평가**: 8.4/10 (투표 수: 31,493)
- **홈페이지**: [Fight Club Official Website](http://www.foxmovies.com/movies/fight-club)

![포스터](https://image.tmdb.org/t/p/w780/pB8BM7pdSp6B6Ih7QZ4DrQ3PmJK.jpg)

- **제작 회사**: Fox 2000 Pictures, Regency Enterprises, Linson Entertainment, 20th Century Fox, Taurus Film

이 영화는 남성성, 소비주의, 정신적 고뇌와 같은 주제를 다루고 있습니다.


In [7]:
run_agent("movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘")

[Tool Call] get_movie_credits({'movie_id': 550})
영화 ID 550에 해당하는 영화인 "Fight Club"의 주요 출연진은 다음과 같습니다:

1. **Edward Norton**
   - 역할: Narrator
   - ![Edward Norton](https://image.tmdb.org/t/p/w185/8nytsqL59SFJTVYVrN72k6qkGgJ.jpg)

2. **Brad Pitt**
   - 역할: Tyler Durden
   - ![Brad Pitt](https://image.tmdb.org/t/p/w185/cckcYc2v0yh1tc9QjRelptcOBko.jpg)

3. **Helena Bonham Carter**
   - 역할: Marla Singer
   - ![Helena Bonham Carter](https://image.tmdb.org/t/p/w185/hJMbNSPJ2PCahsP3rNEU39C8GWU.jpg)

4. **Meat Loaf**
   - 역할: Robert Paulson
   - ![Meat Loaf](https://image.tmdb.org/t/p/w185/7gKLR1u46OB8WJ6m06LemNBCMx6.jpg)

5. **Jared Leto**
   - 역할: Angel Face
   - ![Jared Leto](https://image.tmdb.org/t/p/w185/ca3x0OfIKbJppZh8S1Alx3GfUZO.jpg)

6. **Zach Grenier**
   - 역할: Richard Chesler (Regional Manager)
   - ![Zach Grenier](https://image.tmdb.org/t/p/w185/fSyQKZO39sUsqY283GXiScOg3Hi.jpg)

7. **Holt McCallany**
   - 역할: The Mechanic
   - ![Holt McCallany](https://image.tmdb.org/t/p/w185/iRo9Y